# 迭代器 vs 生成器

本文系统整理 Python 中 **可迭代对象（iterable）**、**迭代器（iterator）**、**生成器（generator）** 的核心概念、运行机制、典型写法与应用场景。

学习目标：

- 理解什么是可迭代对象
- 理解什么是迭代器，以及 `iter()` / `next()` 的作用
- 理解 `for` 循环背后的运行机制
- 掌握自定义迭代器的写法
- 理解生成器函数、`yield`、`yield from`、`send()`
- 掌握迭代器与生成器的区别与适用场景


## 1. 先区分两个概念

在学习迭代器之前，必须先区分：

1. **可迭代对象（iterable）**
2. **迭代器（iterator）**

它们关系密切，但并不是同一个概念。

### 1.1 可迭代对象（iterable）

**概念：** 能被 `for` 循环遍历的对象，就是可迭代对象。

常见的可迭代对象有：

- `list`
- `tuple`
- `str`
- `dict`
- `set`
- `range`
- 文件对象
- 自定义实现了迭代协议的对象


In [1]:
names = ['张三', '李四', '王五']
citys = ('北京', '上海', '深圳')
msg = 'hello'

for item in names:
    print(item)

print('-' * 30)

for item in citys:
    print(item)

print('-' * 30)

for item in msg:
    print(item)


张三
李四
王五
------------------------------
北京
上海
深圳
------------------------------
h
e
l
l
o


下面这些对象通常**不是**可迭代对象：

- 普通整数
- 普通函数对象（默认不能直接被 `for` 遍历）


In [2]:
age = 10

def test():
    pass

print('age 是否可迭代：', hasattr(age, '__iter__'))
print('test 是否可迭代：', hasattr(test, '__iter__'))


age 是否可迭代： False
test 是否可迭代： False


### 1.2 `__iter__` 方法

可迭代对象通常都具备 `__iter__` 方法，因此可以通过 `iter(obj)` 获取与之对应的迭代器。

In [3]:
names = ['张三', '李四', '王五']
citys = ('北京', '上海', '深圳')
msg = 'hello'
age = 10

print(hasattr(names, '__iter__'))
print(hasattr(citys, '__iter__'))
print(hasattr(msg, '__iter__'))
print(hasattr(age, '__iter__'))


True
True
True
False


## 2. 迭代器（iterator）

调用可迭代对象的 `__iter__()` 方法，或者使用 `iter(obj)`，通常会得到一个**迭代器对象**。

### 2.1 关键特点

- 迭代器可以被 `next()` 逐个取值
- 迭代器内部会记录当前遍历状态
- 元素取完后，再次取值会抛出 `StopIteration`
- 迭代器通常是**一次性的**，会被消耗


In [4]:
names = ['张三', '李四', '王五']

print(names.__iter__())
print(iter(names))


In [5]:
names = ['张三', '李四', '王五']
it = iter(names)

print(next(it))
print(next(it))
print(next(it))

try:
    print(next(it))
except StopIteration:
    print('迭代器已经取完，触发 StopIteration')


张三
李四
王五
迭代器已经取完，触发 StopIteration


### 2.2 `for` 循环背后的逻辑

我们平时写：

```python
for item in names:
    print(item)
```

它的底层思想大致等价于：

1. 先调用 `iter(names)` 获取迭代器
2. 再不断调用 `next(it)` 取值
3. 取完后捕获 `StopIteration` 结束循环


In [6]:
names = ['张三', '李四', '王五']

it = iter(names)

while True:
    try:
        item = next(it)
        print(item)
    except StopIteration:
        print('遍历结束')
        break


张三
李四
王五
遍历结束


### 2.3 迭代器本身也有 `__iter__`

迭代器对象不仅有 `__next__`，还必须有 `__iter__`，并且 `iter(迭代器)` 应返回其自身。

这是为了让 `for` 循环在处理迭代器时也能正常工作。

In [7]:
names = ['张三', '李四', '王五']
it = iter(names)

print(it)
print(iter(it))
print(iter(it) is it)


True


### 2.4 迭代器协议

一个对象如果同时满足以下条件，就可以看作迭代器：

- 能被 `iter()` 调用
- 能被 `next()` 逐步取值

也就是说，通常要实现：

- `__iter__()`
- `__next__()`


### 2.5 迭代器会被消耗

迭代器不是可无限重用的容器，而是一个**状态向前推进的对象**。
一旦元素被取完，就不能自动回到开头。

In [8]:
names = ['张三', '李四', '王五']

it1 = iter(names)
it2 = iter(names)

print(next(it1))
print(next(it1))
print(next(it1))

try:
    print(next(it1))
except StopIteration:
    print('it1 已耗尽')

print(next(it2))
print(next(it2))
print(next(it2))


张三
李四
王五
it1 已耗尽
张三
李四
王五


## 3. 自定义迭代器：让 `Person` 支持 `for` 遍历

### 3.1 方案一：可迭代对象和迭代器分离

这种设计更清晰：

- `Person` 负责提供数据
- `PersonIterator` 负责维护遍历状态


In [9]:
class Person:
    def __init__(self, name, age, gender, address):
        self.name = name
        self.age = age
        self.gender = gender
        self.address = address

    def __iter__(self):
        return PersonIterator(self)


class PersonIterator:
    def __init__(self, p):
        self.index = 0
        self.attrs = [p.name, p.age, p.gender, p.address]

    def __iter__(self):
        return self

    def __next__(self):
        if self.index >= len(self.attrs):
            raise StopIteration
        value = self.attrs[self.index]
        self.index += 1
        return value


p1 = Person('张三', 18, '男', '北京昌平')

for item in p1:
    print(item)

print('-' * 30)

for item in p1:
    print(item)


张三
18
男
北京昌平
------------------------------
张三
18
男
北京昌平


### 3.2 方案二：对象本身既是可迭代对象，又是迭代器

这种写法更简洁，但需要注意：

- 对象内部要自己维护迭代状态
- 每次开始遍历前最好重置状态


In [10]:
class Person:
    def __init__(self, name, age, gender, address):
        self.name = name
        self.age = age
        self.gender = gender
        self.address = address
        self.__index = 0
        self.__attrs = [name, age, gender, address]

    def __iter__(self):
        self.__index = 0
        return self

    def __next__(self):
        if self.__index >= len(self.__attrs):
            raise StopIteration
        value = self.__attrs[self.__index]
        self.__index += 1
        return value


p1 = Person('张三', 18, '男', '北京昌平')

for item in p1:
    print(item)

print('-' * 30)

for item in p1:
    print(item)


张三
18
男
北京昌平
------------------------------
张三
18
男
北京昌平


### 3.3 进阶：在 `__next__` 中加工数据

迭代器真正的核心在于 `__next__()`：

- 可以决定返回什么
- 可以控制状态如何推进
- 可以在返回前对数据做转换

下面示例中：

- 字符串统一转为大写
- 整数转为中文数字（使用第三方库 `cn2an`）


In [11]:
# 如未安装，请先执行：
# !pip install cn2an

from cn2an import an2cn

class Person:
    def __init__(self, name, age, gender, address):
        self.name = name
        self.age = age
        self.gender = gender
        self.address = address
        self.__index = 0
        self.__attrs = [name, age, gender, address]

    def __iter__(self):
        self.__index = 0
        return self

    def __next__(self):
        if self.__index >= len(self.__attrs):
            raise StopIteration

        value = self.__attrs[self.__index]

        if isinstance(value, str):
            value = value.upper()
        if isinstance(value, int):
            value = an2cn(value)

        self.__index += 1
        return value


p1 = Person('zhangsan', 18, '男', '北京昌平')

for item in p1:
    print(item)


ZHANGSAN
十八
男
北京昌平


## 4. 迭代器的优势

迭代器最大的优势是：**惰性计算（lazy evaluation）**。

也就是说：

- 需要一个元素，就生成一个元素
- 不会一次性把所有结果都创建出来
- 当数据量很大时，内存更节省

这非常适合：

- 大文件逐行读取
- 海量数据处理
- 网络数据流
- 只需要部分结果的场景


### 4.1 用迭代器实现斐波那契数列

In [12]:
class Fibo:
    def __init__(self, total):
        self.total = total
        self.index = 0
        self.pre = 1
        self.cur = 1

    def __iter__(self):
        return self

    def __next__(self):
        if self.index >= self.total:
            raise StopIteration

        if self.index < 2:
            value = 1
        else:
            value = self.pre + self.cur
            self.pre = self.cur
            self.cur = value

        self.index += 1
        return value


for item in Fibo(10):
    print(item)


1
1
2
3
5
8
13
21
34
55


### 4.2 不使用迭代器，直接返回列表

In [13]:
def fibo(total):
    if total <= 0:
        return []
    if total == 1:
        return [1]

    nums = [1, 1]
    for i in range(2, total):
        nums.append(nums[-1] + nums[-2])
    return nums


print(fibo(10))


[1, 1, 2, 3, 5, 8, 13, 21, 34, 55]


### 4.3 简单比较内存占用

下面示例使用 `tracemalloc` 粗略查看内存使用情况。

> 注意：实际结果会受到运行环境、解释器状态、对象生命周期等影响，因此这里只做学习演示。

In [14]:
import tracemalloc

tracemalloc.start()
f1 = Fibo(100000)
m1 = tracemalloc.get_traced_memory()[1]
tracemalloc.stop()

tracemalloc.start()
f2 = fibo(100000)
m2 = tracemalloc.get_traced_memory()[1]
tracemalloc.stop()

print(f'迭代器对象峰值内存：{m1 / 1024 / 1024:.4f} MB')
print(f'列表结果峰值内存：{m2 / 1024 / 1024:.4f} MB')


迭代器对象峰值内存：0.0188 MB
列表结果峰值内存：445.0114 MB


## 5. 生成器（generator）

生成器可以看作是**一种更方便的迭代器实现方式**。

只要函数体中出现了 `yield`，这个函数就是**生成器函数**。
调用它时，函数不会立刻执行，而是返回一个**生成器对象**。

### 5.1 两个核心概念

- **生成器函数**：函数中出现 `yield`
- **生成器对象**：调用生成器函数后得到的对象

即使 `yield` 在某个分支里暂时走不到，只要函数体内出现过 `yield`，它就是生成器函数。

In [15]:
def demo():
    print('demo 函数开始执行')
    print(100)
    yield
    a = 200
    print(a)

d = demo()
print(d)


<generator object demo at 0x000001D4A03B4B80>


### 5.2 `yield` 的执行过程

当生成器对象被 `next()` 调用时：

1. 函数开始执行
2. 运行到 `yield` 时暂停
3. `yield` 后面的值会作为本次 `next()` 的返回值
4. 下一次再调用 `next()` 时，从暂停位置继续执行
5. 如果运行到 `return` 或函数自然结束，则抛出 `StopIteration`


In [16]:
def demo():
    print('demo 函数开始执行')
    print(100)
    yield '第1次 yield 返回的数据'
    a = 200
    print(a)
    yield '第2次 yield 返回的数据'
    b = 300
    print(b)
    return '生成器执行结束'

d = demo()

r1 = next(d)
print(r1)

r2 = next(d)
print(r2)

try:
    next(d)
except StopIteration as e:
    print('StopIteration 携带的信息：', e)


demo 函数开始执行
100
第1次 yield 返回的数据
200
第2次 yield 返回的数据
300
StopIteration 携带的信息： 生成器执行结束


### 5.3 生成器对象本质上也是迭代器

生成器对象天然支持：

- `__iter__`
- `__next__`

因此它本质上就是一种特殊的迭代器。

In [17]:
def demo():
    yield 'A'
    yield 'B'

d = demo()

print(hasattr(d, '__iter__'))
print(hasattr(d, '__next__'))
print(iter(d) is d)

for item in d:
    print(item)


True
True
True
A
B


### 5.4 `yield` 可以写在循环中

这是生成器最常见的用法之一：逐个生成数据。

In [18]:
def create_car(total):
    for index in range(1, total + 1):
        yield f'第{index}台车'

cars = create_car(5)

print(next(cars))
print(next(cars))
print(next(cars))
print(next(cars))
print(next(cars))


第1台车
第2台车
第3台车
第4台车
第5台车


### 5.5 `yield from`

`yield from 可迭代对象` 可以把该对象中的元素依次产出。

它常常可以替代：

```python
for item in obj:
    yield item
```

In [19]:
def demo():
    nums = [10, 20, 30, 40]
    yield from nums

d = demo()

print(next(d))
print(next(d))
print(next(d))
print(next(d))


10
20
30
40


### 5.6 `send()`：一边继续执行，一边给生成器传值

`send(value)` 比 `next()` 更强大：

- `next()`：只能取值
- `send(value)`：既能恢复执行，又能给上一个 `yield` 表达式传值

注意：**第一次启动生成器时不能发送非 `None` 的值**。

In [20]:
def demo():
    print('demo 函数开始执行')
    print(100)
    a = yield '第1次 yield 返回的数据'
    print('a =', a)
    b = yield '第2次 yield 返回的数据'
    print('b =', b)
    return '执行完成'

d = demo()

r1 = next(d)   # 等价于 d.send(None)
print(r1)

r2 = d.send(666)
print(r2)

try:
    d.send(888)
except StopIteration as e:
    print('StopIteration 携带的信息：', e)


demo 函数开始执行
100
第1次 yield 返回的数据
a = 666
第2次 yield 返回的数据
b = 888
StopIteration 携带的信息： 执行完成


## 6. 生成器的应用

### 6.1 用生成器遍历 `Person` 对象

相比手写 `__next__`，生成器写法通常更简洁。

In [21]:
class Person:
    def __init__(self, name, age, gender, address):
        self.name = name
        self.age = age
        self.gender = gender
        self.address = address
        self.__attrs = [name, age, gender, address]

    def __iter__(self):
        yield from self.__attrs


p1 = Person('张三', 18, '男', '北京昌平')

for attr in p1:
    print(attr)


张三
18
男
北京昌平


### 6.2 用生成器实现斐波那契数列

In [22]:
def fibo(total):
    pre = 1
    cur = 1
    for index in range(total):
        if index < 2:
            yield 1
        else:
            value = pre + cur
            pre = cur
            cur = value
            yield value


for item in fibo(10):
    print(item)


1
1
2
3
5
8
13
21
34
55


### 6.3 生成器也能被一次性展开

无论是迭代器还是生成器，都可以被 `list()`、`tuple()`、`set()` 等直接取出全部内容。

但需要注意：

- 一旦展开，惰性优势就没了
- 数据量大时可能造成较高内存占用
- 展开后生成器会被消耗


In [23]:
def fibo(total):
    pre = 1
    cur = 1
    for index in range(total):
        if index < 2:
            yield 1
        else:
            value = pre + cur
            pre = cur
            cur = value
            yield value

f1 = fibo(10)
result = list(f1)
print(result)


[1, 1, 2, 3, 5, 8, 13, 21, 34, 55]


## 7. 生成器表达式

生成器表达式的语法和列表推导式很像，但它不会立刻生成所有结果。

### 语法

```python
(表达式 for 变量 in 可迭代对象)
```

适合场景：

- 每个结果只依赖当前元素
- 想节省内存
- 不需要一次性拿到所有结果


In [24]:
nums = [10, 20, 30, 40]

result1 = [n * 2 for n in nums]
print('列表推导式：', result1)

result2 = (n * 2 for n in nums)
print('生成器表达式：', result2)

for item in result2:
    print(item)


列表推导式： [20, 40, 60, 80]
生成器表达式： <generator object <genexpr> at 0x000001D4A02E4790>
20
40
60
80


## 8. 迭代器与生成器对比总结

| 维度 | 迭代器 | 生成器 |
|---|---|---|
| 本质 | 遵循迭代器协议的对象 | 一种特殊的迭代器 |
| 创建方式 | 手动实现 `__iter__` / `__next__` | 函数中使用 `yield` |
| 写法复杂度 | 通常更高 | 通常更简洁 |
| 状态维护 | 需要手动维护 | Python 自动保存执行状态 |
| 惰性计算 | 支持 | 支持 |
| 适合场景 | 需要精细控制迭代逻辑 | 逐步产出数据、代码简洁优先 |


## 9. 易错点整理

1. **可迭代对象不一定是迭代器**
   - 列表、字符串通常是可迭代对象
   - `iter(list_obj)` 的结果才是迭代器

2. **迭代器会被消耗**
   - 一旦遍历完，再次 `next()` 会抛出 `StopIteration`

3. **生成器也是迭代器**
   - 所以生成器同样会被消耗

4. **第一次启动生成器不能 `send(非None值)`**
   - 应先用 `next(gen)` 或 `gen.send(None)` 启动

5. **不要把生成器表达式和列表推导式混淆**
   - `[]` 得到列表
   - `()` 得到生成器对象


## 10. 最后总结

可以用一句话概括：

- **可迭代对象**：能被 `for` 遍历
- **迭代器**：能被 `next()` 逐个取值
- **生成器**：用 `yield` 写出来的“自动版迭代器”

实战建议：

1. 如果只是正常遍历数据，直接使用 `for`
2. 如果需要自定义遍历规则，可手写迭代器
3. 如果想简洁地实现惰性迭代，优先考虑生成器
4. 如果数据量很大，尽量避免一次性转成列表
